# NB_GLD_SRV_POI_REVIEW

RECOMPUTE `lh_vv_gold.dbo.gld_srv_poi_review`. Grain một dòng một `review_id`.
`slv_enr_poi_review_multi_lang` INNER JOIN `slv_3p_poi_review` theo `review_id`.
LEFT JOIN `slv_3p_poi_rating` theo `poi_id` của enrich. Trùng `poi_id` thì dừng, không chọn dòng.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
ML_COLS = ("review_id", "poi_id", "lang", "title", "review_text")
REVIEW_COLS = ("review_id", "poi_id", "rating", "time", "author_name")
RATING_COLS = ("poi_id", "rating_overall", "rating_count")
SOURCE_TYPES = {
    "slv_3p_poi_review": {"rating": "decimal(4,2)"},
    "slv_3p_poi_rating": {"rating_overall": "decimal(4,2)", "rating_count": "bigint"},
}
OUT_COLS = ("review_id", "poi_id", "lang", "rating_overall", "rating_count", "title", "review_text",
            "rating", "review_date", "author_name")


def _require(df, cols, name, hint=""):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ConfigError(f"gld_srv_poi_review: {name} thiếu {missing}{hint}")


def _types(df, name):
    got = dict(df.dtypes)
    bad = {c: got[c] for c, t in SOURCE_TYPES[name].items() if got[c] != t}
    if bad:
        raise ConfigError(f"gld_srv_poi_review: {name} sai kiểu {bad}, cần {SOURCE_TYPES[name]}")


def build(ctx):
    ml = ctx.input("multi_lang", live_only=False)
    rv = ctx.input("review")
    rt = ctx.input("rating")
    _require(ml, ML_COLS, "slv_enr_poi_review_multi_lang", ": chạy srvrevpy của NB_CREATE_DDL")
    _require(rv, REVIEW_COLS, "slv_3p_poi_review")
    _require(rt, RATING_COLS, "slv_3p_poi_rating")
    _types(rv, "slv_3p_poi_review")
    _types(rt, "slv_3p_poi_rating")
    # Một dòng mỗi poi_id. Trùng poi_id -> dừng, không chọn dòng.
    rt = rt.select(*RATING_COLS)
    dup = rt.where(F.col("poi_id").isNotNull()).groupBy("poi_id").count().where("count > 1").count()
    if dup:
        raise QualityGateError(f"gld_srv_poi_review: slv_3p_poi_rating {dup} poi_id có hơn một dòng")
    review = rv.select("review_id", F.col("poi_id").alias("_rv_poi"), "rating",
                       F.col("time").alias("review_date"), "author_name")
    ml = ml.select(*ML_COLS)
    ctx.warn("enrich_without_review", ml.join(review, "review_id", "left_anti").count())
    rows = (ml.join(review, "review_id", "inner")
              .join(rt.withColumn("_rated", F.lit(True)), "poi_id", "left"))
    ctx.warn("review_poi_id_mismatch", rows.where(~F.col("poi_id").eqNullSafe(F.col("_rv_poi"))).count())
    ctx.metrics.update(rating_matched=rows.where("_rated IS NOT NULL").count())
    return rows.select(*OUT_COLS)


SPEC = NodeSpec(
    trg_tbl="gld_srv_poi_review",
    inputs={"multi_lang": "slv_enr_poi_review_multi_lang", "review": "slv_3p_poi_review",
            "rating": "slv_3p_poi_rating"},
    pk=("review_id",),
    build=build,
    required_inputs=("multi_lang", "review"),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
